In [1]:
%reset -f
import warnings
warnings.filterwarnings('ignore')

import numpy as np

import pandas as pd
pd.options.mode.use_inf_as_na = True # this is instead of inf to show NAs 

import time 
import os 
from os import path

import itertools
from itertools import chain

import datetime as dt
from datetime import date, timedelta

from timebudget import timebudget

import ray 

import ffn

import matplotlib.pyplot as plt
import pyfolio as pf  # you should download the pyfolio-reloaded 

import pypfopt
from pypfopt import expected_returns, EfficientCVaR, CLA
from pypfopt import HRPOpt    # Hierarchical Parity Optimization 
from pypfopt import plotting


In [2]:
%cd /Users/safishajjouz/GitHub/QuantitativePortfolioManagement

/Users/safishajjouz/GitHub/QuantitativePortfolioManagement


In [3]:
from myPortfolioManagement.myData import get_stock_prices, data_overview
from myPortfolioManagement.myPortfolioSelection import possible_combinations

#### Load Prices 

In [4]:
# import list of funds 
mypath    = '/Users/safishajjouz/GitHub/QuantitativePortfolioManagement/myPortfolioManagement/Data'
file_name = "list_of_funds.xlsx"
file_path_to_load = path.join(mypath, file_name)
df = pd.read_excel(file_path_to_load)

# cleam 
df = df[(df['my_portfolio']=='T') | (df['market_neutral']=='T') ]
df = df.dropna(subset =['yahooTicker'])
df = df [['short_name', 'ISIN', 'yahooTicker']]

yahoo_tickers = df['yahooTicker'].to_list()

# Set dates 
start_date = '1980-01-01'
end_date = date.today() -  timedelta(days=1)
end_date = end_date.strftime("%Y-%m-%d") 


df_prices = get_stock_prices(yahoo_tickers = yahoo_tickers,
                              start_date = start_date, 
                              end_date = end_date, wide_format = False)

get_stock_prices took 4.436sec


In [5]:
df_prices_long = df_prices.reset_index().merge(df[['short_name', 'yahooTicker']], on = 'yahooTicker').set_index('Date')
df_prices_long = df_prices_long[['adjclose', 'short_name', 'yahooTicker']]

# data_overview(df_prices_long, my_assets_col_name = 'short_name', my_date_col_name = 'Date')

df_prices_wide =  df_prices_long.reset_index().pivot_table(index = 'Date',
                    columns='short_name', 
                    values='adjclose')

### Clean Data Frame 

In [6]:

df_prices_wide = df_prices_wide[df_prices_wide.index>='2018-01-02']

# full Dataframe with Returns Cleaned (denoise returns)
df_cleaned_ma =df_prices_wide.rolling(window=50, min_periods=1).mean()
df_cleaned_ma = df_cleaned_ma.dropna(axis=0, how='all')

### Calculate returns 

In [7]:
# calculate average returns based on geometric average  
retunrs_average = expected_returns.mean_historical_return(df_prices_wide, 
                                        returns_data=False, 
                                        compounding=True, # computes geometric average 
                                        frequency=252)   # frequency to compute annualised returns 

# average returns based on ema 
returns_average_ema = pypfopt.expected_returns.ema_historical_return(df_prices_wide, 
                                               returns_data=False, 
                                               compounding=True, 
                                               span=500,  # days to calculate EMA 
                                               frequency=252)

# calculate CAPM returns 
returns_capm = expected_returns.capm_return(df_cleaned_ma.drop('World ex-UK', axis = 1), 
                             market_prices=df_cleaned_ma[['World ex-UK']],  # price of benchmark 
                             returns_data=False, 
                             risk_free_rate=0.02, 
                             compounding=True, 
                             frequency=252)

# daily return 
ret = pypfopt.expected_returns.returns_from_prices(df_cleaned_ma)

# full Dataframe with Returns Cleaned
#rets_all = expected_returns.returns_from_prices(df_cleaned_ma.interpolate(method = "time"))
#rets_all.index = rets_all.index.tz_localize('utc')
#rets_all 
# pf.timeseries.cum_returns(rets_all[['SMT', 'FCSS', 'World ex-UK', 'BG Pacific', 'JM Global Abs Returns']], starting_value=1).plot(figsize=(10,10))

In [8]:
# Cluster funds  
ffn.core.calc_ftca(ret, threshold=0.70)

{1: ['Rathbone Multi-Asset',
  'ASI Small Cap',
  'BG Income',
  'BG Pacific',
  'Core S&P Small-Cap',
  'Fidelity Global Sit',
  'Fundsmith',
  'JH European Abs Returns',
  'MI UK Growth',
  'Pacific Horizon',
  'RL Diversified',
  'Rathbone Ethical Bonds',
  'Rathbone Global',
  'SMT',
  'Stewart Asia Leaders',
  'Throgmorton Trust',
  'Trojan',
  'Vanguard Global Small Cap',
  'World ex-UK'],
 2: ['Gold'],
 3: ['BG Positive Change', 'BG Discovery'],
 4: ['JM Global Abs Returns'],
 5: ['Sanlam Multi-Strategy',
  'Aegon Market Neutral',
  'JH Abs Returns',
  'Jupiter Absolute Return Bond'],
 6: ['Allianz Technology'],
 7: ['ASI Abs Returns'],
 8: ['RL Yield Bond'],
 9: ['Merian Strategic Bond'],
 10: ['BLR Mining'],
 11: ['JPM Global Macro'],
 12: ['Allianz Strategic Bond']}

In [9]:
# determine optimum number of different asset classes  
N = len(ffn.core.calc_ftca(ret, threshold=0.70))
N

12

In [ ]:
# slow process 

asset_to_consider = df_prices_long.short_name.unique()
all_portf = possible_combinations(asset_to_consider, min_assets = 3, max_assets = 6)
all_portf

In [ ]:
# legacy name (lazy to change the code)
all_portf_new = all_portf

training_start_date = '2018-01-02'
training_end_date = "2021-03-01" # end of sample 


all_portfolios_in_lists = [[*all_portf_new[i]] for i in range(len(all_portf_new))]

def portfolio_optimisation_routine(rets):
    
    #optimize portfolio HRP 
    hrp = HRPOpt(rets)
    hrp.optimize()
    weights = hrp.clean_weights()
    df_weights = pd.DataFrame(weights , index=[0])
    df_weights = pd.melt(df_weights, var_name = 'fund', value_name = 'weight')
    perf_temp = round(hrp.portfolio_performance(verbose=False)[2], 2)
    
    # collect results 
    results_temp = (df_weights, perf_temp)
    return results_temp



In [ ]:
@ray.remote
def portfolio_selection_routine(asset_prices_wide, 
                                training_start_date:str, 
                                training_end_date:str, 
                                list_of_assets:list):
    
    # slice columns 
    asset_prices = asset_prices_wide[list_of_assets]
    
    # slice index Date 
    asset_prices =  asset_prices[(asset_prices.index>= training_start_date) & 
                                 (asset_prices.index<= training_end_date)]
    
    # calculate average returns based on geometric average  
    retunrs_average = expected_returns.returns_from_prices(asset_prices) 
    
    # optimise portfolio
    results_temp = portfolio_optimisation_routine(retunrs_average)
    
    return results_temp
    

In [ ]:
training_start_date = '2018-01-02'
training_end_date = "2021-03-01" # end of sample 

portfolio_selection_routine(asset_prices_wide = df_cleaned_ma, 
                                training_start_date = training_start_date, 
                                training_end_date = training_end_date, 
                                list_of_assets = all_portfolios_in_lists[0])

In [ ]:
# loop over tickers block 
ray.init(ignore_reinit_error=True, num_cpus=5)
@timebudget
def loop_over_portfolios(operation,asset_prices_wide, training_start_date, training_end_date,input):
    portfolio_opt = ray.get([operation.remote(asset_prices_wide,
                                              training_start_date,training_end_date,portfolio) for portfolio in input])
    return portfolio_opt

portfolios_to_check=loop_over_portfolios(portfolio_selection_routine,df_cleaned_ma,
                            training_start_date, training_end_date, all_portfolios_in_lists)
ray.shutdown()
    

In [ ]:
## Model Selection Results 

# sort results for HRP 
portfolio_selection_results = sorted(portfolios_to_check, 
                                     key=lambda tup: tup[1])
portfolio_selection_results.reverse() 
portfolio_selection_results

In [ ]:
# remove highly concentrated portfolios and dataframe 

## HRP portfolios 
portfolios_final = []
for i in range(0,len(portfolio_selection_results)):
    # max constraint 
    df_select_temp = portfolio_selection_results[i][0]
    if (max(df_select_temp["weight"])<=0.20):
        df_select = df_select_temp
        df_select["portf_position"]= "portfolio" + str(i)
        df_select["portf_sharp"] = portfolio_selection_results[i][1]
        portfolios_final.append(df_select)

portfolios_to_consider = pd.concat(portfolios_final)
max(portfolios_to_consider["weight"])

In [ ]:
portfolios_to_consider = pd.concat([portfolios_to_consider])

In [ ]:
# Filter Portfolios 

#calculate portfolio returns 
def cul_port_returns(df_returns,df_weights, portfolio_name = None):
    '''
    df_returns: a wide DataFrame series with assets returns 
    df_wegiths: a long Datafrane with Fund names in the first colum 
                and weight in the other 
    '''
    df_weights = df_weights.sort_values("fund")
    funds = list(df_weights["fund"])
    df_returns = df_returns[funds]
    df_returns =  df_returns[sorted(df_returns.columns)]
    weights = pd.Series.to_numpy(df_weights["weight"])
    port_returns = df_returns.fillna(0).dot(weights)  # this is a workaround due to NAs 
    if portfolio_name is None: 
        port_returns = pd.Series(port_returns, name = "portfolio")
    else:
        port_returns = pd.Series(port_returns, name = portfolio_name)
    port_returns = pd.DataFrame(port_returns)
    return port_returns

port_list = []
print("first loop")
for port in portfolios_final:
    port_name = port["portf_position"].drop_duplicates()[0]
    port =  port[["fund", "weight"]]
    rets_temp = cul_port_returns(ret,port, portfolio_name= str(port_name))
    port_list.append(rets_temp)

In [ ]:
portf_returns = pd.concat(port_list, axis=1)
portf_returns

In [ ]:
from scipy import stats as scipy_stats
import numpy as np

def probabilistic_sharpe_ratio(returns, sr_benchmark=0.0):
    """
    Calculate the Probabilistic Sharpe Ratio (PSR).
    Parameters
    ----------
    returns: np.array, pd.Series, pd.DataFrame
        If no `returns` are passed it is mandatory to pass a `sr` and `sr_std`.
    sr_benchmark: float
        Benchmark sharpe ratio expressed in the same frequency as the other parameters.
        By default set to zero (comparing against no investment skill).
    sr: float, np.array, pd.Series, pd.DataFrame
        Sharpe ratio expressed in the same frequency as the other parameters.
    sr_std: float, np.array, pd.Series, pd.DataFrame
        Standard deviation fo the Estimated sharpe ratio,
        expressed in the same frequency as the other parameters.
    Returns
    -------
    float, pd.Series
    Notes
    -----
    PSR(SR*) = probability that SR^ > SR*
    SR^ = sharpe ratio estimated with `returns`, or `sr`
    SR* = `sr_benchmark`
    https://papers.ssrn.com/sol3/papers.cfm?abstract_id=1821643
    """
    
    ## TO DO check with annualized Returns 
    
    T = len(returns)    
    kr = pd.Series(scipy_stats.kurtosis(returns, fisher=False), index=returns.columns)
    sk = pd.Series(scipy_stats.skew(returns), index=returns.columns) # this should be set to zero for small samples 
    sr = pd.Series(returns.mean() / returns.std(ddof=1), index = returns.columns) # sharp non-annualized 
    numer = (sr - sr_benchmark)*((T-1)**0.5)
    den = (1 - sk*sr + ((kr-1)*sr**2)/4 )**0.5
    psr = scipy_stats.norm.cdf(numer/den)

    if type(returns) == pd.DataFrame:
        psr = pd.Series(psr, index=returns.columns)
    elif type(psr) not in (float, np.float64):
        psr = psr[0]

    return psr



In [ ]:
psr = probabilistic_sharpe_ratio(portf_returns, sr_benchmark=0.20)
psr = psr[psr.sort_values(ascending=False)>=0.95]
psr

In [ ]:
rets_all.columns

In [ ]:
# check among those portfolios which ones might beat the Benchmark 
# get the sharp value (non-annualized of the Benchmark)
rets_benchmark = ret["World ex-UK"]
sr_b = pd.Series(rets_benchmark.mean() / rets_benchmark.std(ddof=1))
psr2 = probabilistic_sharpe_ratio(portf_returns[psr.index], sr_benchmark=sr_b[0])
if (psr2[psr2.sort_values(ascending=False)>=0.90]).empty:
    print("No portoflio likely to give better Sharpe than Benchmark")
    print("Consider No Investment skill, other portfolio construction, or look to allocate the money in the benachmark")
else:
    psr2.sort_values(ascending=False)

In [ ]:
# check among those portfolios which ones might beat the Benchmark in out of sample 

out_of_sample_date = "2021-03-01"

# get the sharp value (non-annualized of the Benchmark)
rets_benchmark = ret["World ex-UK"]
rets_benchmark = rets_benchmark[rets_benchmark.index>=out_of_sample_date ]
sr_b_out_of_sample = pd.Series(rets_benchmark.mean() / rets_benchmark.std(ddof=1))


# out_of_sample 
portf_returns_out_of_sample = portf_returns[psr.index]
portf_returns_out_of_sample = portf_returns_out_of_sample[portf_returns_out_of_sample.index>= out_of_sample_date]
psr3 = probabilistic_sharpe_ratio(portf_returns_out_of_sample, sr_benchmark=sr_b_out_of_sample[0])


In [ ]:
psr3.sort_values(ascending=False)

In [ ]:
# pick 
psr_top = psr.sort_values(ascending= False).head(5)
psr_top_oos1 = psr2[psr2>=0.85]
psr_top_oos2 = psr3[psr3>=0.85]

In [ ]:
list(set(psr_top_oos2.index.to_list() + psr_top_oos1.index.to_list() + psr_top.index.to_list() ))

In [ ]:
returns_best_portfolios = portf_returns[list(set(psr_top_oos2.index.to_list() + psr_top_oos1.index.to_list() + psr_top.index.to_list() ))]
returns_best_portfolios

In [ ]:
# get the portfolios with the most stable Sortino ratios 
returns_best_portfolios_out_of_sample = returns_best_portfolios[returns_best_portfolios.index>= out_of_sample_date]
returns_best_portfolios_training_period = returns_best_portfolios[returns_best_portfolios.index<out_of_sample_date ]

df_srt_out_smpl = pf.timeseries.sortino_ratio(returns_best_portfolios_out_of_sample)
df_srt_out_smpl.index = returns_best_portfolios_out_of_sample.columns
df_srt_out_smpl = pd.Series(df_srt_out_smpl, name = "Sortino_out_of_sample").to_frame()

df_srt_in_smpl = pf.timeseries.sortino_ratio(returns_best_portfolios_training_period)
df_srt_in_smpl.index = returns_best_portfolios_training_period.columns
df_srt_in_smpl = pd.Series(df_srt_in_smpl, name = "Sortino_in_sample").to_frame()

df_srt = pd.concat([df_srt_in_smpl, df_srt_out_smpl], axis = 1)
df_srt["Diff"] = df_srt["Sortino_out_of_sample"] - df_srt["Sortino_in_sample"]

top_portfolios = df_srt.sort_values("Diff").head(10)
top_portfolios

In [ ]:
stable_portfolios = top_portfolios[(top_portfolios['Diff']>=-0.1) & (top_portfolios['Diff']<=6)]
stable_portfolios

In [ ]:
price_index = ffn.core.to_price_index(returns_best_portfolios[stable_portfolios.index], start=100)
perf = price_index.calc_stats()
print(perf.display())

In [ ]:
pf.timeseries.cum_returns(returns_best_portfolios, starting_value=1).plot(figsize=(10,10))

In [ ]:
# check some basic perfomance stats for each asset 
price_index = ffn.core.to_price_index(returns_best_portfolios, start=100)
df_stats = ffn.core.GroupStats(price_index).stats
df_stats.dropna()
keep_index = ["total_return", "max_drawdown", "calmar", "ytd", 
              "three_month", "six_month", 'one_year', 'three_year',
              'monthly_sortino']

In [ ]:
df_stats[df_stats.index.isin(keep_index)].transpose().sort_values(by=['monthly_sortino'], ascending= False)

In [ ]:
from sklearn.preprocessing import RobustScaler
scaler = RobustScaler()
df_rank = df_stats[df_stats.index.isin(keep_index)].transpose().sort_values(by=['three_year'], ascending= False)
df_rank = pd.DataFrame(scaler.fit_transform(df_rank), columns=df_rank.columns, index = df_rank.index)
df_rank["ranking_index"] = df_rank.sum(axis=1)
df_rank = df_rank.sort_values(by=['ranking_index'], ascending= False)

In [ ]:
df_rank

In [ ]:
# criteria for the best portfolio 
best_portfolio = df_rank.head(1).index[0]
#best_portfolio = top_portfolios.head(1)
best_portfolio

In [ ]:
df_best_port = portfolios_to_consider[portfolios_to_consider["portf_position"] == best_portfolio]

wealth = 30000
df_best_port["invest"] = df_best_port["weight"]*wealth

In [ ]:
returns_best_portfolios[[best_portfolio]]

In [ ]:
psr_b = ret["World ex-UK"].mean()/ret["World ex-UK"].std()
df_best_port["psr"] = probabilistic_sharpe_ratio(returns_best_portfolios[[best_portfolio]], sr_benchmark=0)[0]
df_best_port["psr_to_bench"] = probabilistic_sharpe_ratio(returns_best_portfolios[[best_portfolio]], sr_benchmark=psr_b)[0]
df_best_port

In [ ]:
probabilistic_sharpe_ratio(returns_best_portfolios[[best_portfolio]], sr_benchmark=psr_b)

In [ ]:
returns = returns_best_portfolios[[best_portfolio]].join(ret["World ex-UK"], on = "Date")
returns = returns[returns.index>=out_of_sample_date]
pf.timeseries.cum_returns(returns, starting_value=1).plot(figsize=(10,10))

In [ ]:
# check some basic perfomance stats for each asset 
price_index = ffn.core.to_price_index(returns, start=100)
df_stats = ffn.core.GroupStats(price_index).stats
df_stats.dropna()
keep_index = ["total_return", "max_drawdown", "calmar", "ytd", 
              "three_month", "six_month", 'one_year', 'three_year',
              'monthly_sortino']
df_stats[df_stats.index.isin(keep_index)].transpose().sort_values(by=['monthly_sortino'], ascending= False)

In [ ]:
#df_best_port.to_csv("best_portf2.csv", index = False, header=True)

In [ ]:
returns_best_portfolios.index
#ret.index

In [ ]:
returns_assets = ret.index.tz_localize('utc')
returns_best_port = returns_best_portfolios.index.tz_localize('utc')

In [ ]:
returns_assets =ret
returns_best_port = returns_best_portfolios
returns_assets.index = ret.index.tz_localize('utc')
returns_best_port.index = returns_best_portfolios.index.tz_localize('utc')

In [ ]:
pf.create_returns_tear_sheet(returns_best_portfolios[best_portfolio], benchmark_rets = returns_assets['World ex-UK'],
                             live_start_date='2021-03-01', return_fig=False)
